# Best predictors based on discrimination, permutation

## 0. Package loading and installation

In [31]:
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
from rpy2.robjects import r, pandas2ri

!pip install scikit-survival # Install scikit-survival if not already installed
!pip install miceforest --no-cache-dir
!pip install lifelines

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

pandas2ri.activate()

#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")


In [32]:
from google.colab import userdata

# Names of the objects (and secrets)
object_names = [
    "times_eval_death",
    "times_eval_readm",
    "imputation_1"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}.parquet"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

  # Names of the objects (and secrets)
object_names = [
    "X_reduced_list.npz.gpg"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(


In [33]:
from google.colab import userdata
import subprocess
import os
import sys

# 1. Create dedicated writable directory
output_path = '/content/X_reduced_list.npz'

# 2. Get passphrase with special character handling
passphrase = userdata.get('passphrase')
if not passphrase:
    raise ValueError("❌ Passphrase not set in Colab userdata. Set via: userdata.set('passphrase', 'your_pass')")

# 3. Decrypt with explicit binary handling
try:
    result = subprocess.run([
        'gpg', '--batch',
        '--yes',  # Overwrite files without confirmation
        '--pinentry-mode', 'loopback',
        '--passphrase', passphrase,
        '--output', output_path,
        '--decrypt', 'X_reduced_list.npz.gpg'
    ],
    capture_output=True,
    text=True,
    check=True  # Raise exception on failure
    )

    print(f"✅ SUCCESS: Decrypted file saved to:\n{output_path}")
    print(f"📦 File size: {os.path.getsize(output_path) / 1024:.1f} KB")

except subprocess.CalledProcessError as e:
    print(f"❌ GPG ERROR (exit code {e.returncode}):")
    stderr = e.stderr.strip()

    # Special handling for common errors
    if "Bad session key" in stderr:
        print("🔑 PASSPHRASE MISMATCH: Verify your passphrase in Colab userdata")
        print("   → Set correct passphrase with: userdata.set('passphrase', 'CORRECT_PASS')")
    elif "No such file" in stderr:
        print("📂 MISSING ENCRYPTED FILE: Upload X_reduced_list.npz.gpg to Colab first")
        print("   → Use left sidebar '📁 Files' tab to upload")
    else:
        print(f"📄 Raw GPG error:\n{stderr}")

    # Show debug info
    print("\n🔍 DEBUG INFO:")
    print(f"- Current directory: {os.getcwd()}")
    print(f"- Files in directory: {os.listdir()}")
    print(f"- Passphrase length: {len(passphrase)} characters (hidden for security)")
    sys.exit(1)

✅ SUCCESS: Decrypted file saved to:
/content/X_reduced_list.npz
📦 File size: 23437.3 KB


Load in python

In [34]:
# ---- Time grids ----
times_eval_death = pd.read_parquet("times_eval_death.parquet")["time"].to_numpy()
times_eval_readm = pd.read_parquet("times_eval_readm.parquet")["time"].to_numpy()

print("times_eval_death shape:", times_eval_death.shape)
print("times_eval_readm shape:", times_eval_readm.shape)

times_eval_death shape: (49,)
times_eval_readm shape: (50,)


Added outcomes




In [35]:
import pandas as pd
import numpy as np
from google.colab import userdata

# ---- Download y_surv_readm & y_surv_death ----
for name in ["y_surv_readm", "y_surv_death"]:
    file_id = userdata.get(name)
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")
    !gdown --id {file_id} --output {name}.parquet --quiet
    print(f"Downloaded {name}.parquet")

# ---- Load and reconstruct Surv-like structured arrays ----
df_y_readm = pd.read_parquet("y_surv_readm.parquet")
y_surv_readm = np.array(
    list(zip(df_y_readm["event"].astype(bool),
             df_y_readm["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

df_y_death = pd.read_parquet("y_surv_death.parquet")
y_surv_death = np.array(
    list(zip(df_y_death["event"].astype(bool),
             df_y_death["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

print("y_surv_readm shape:", y_surv_readm.shape)
print("y_surv_death shape:", y_surv_death.shape)


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_readm.parquet
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_death.parquet
y_surv_readm shape: (88504,)
y_surv_death shape: (88504,)


### Retrieve the list and Get variable names

From imputation_1, we will get the names of each feature to discard those that not contrbitute to the predictors.

In [36]:
import numpy as np
import pandas as pd

# Load the data
data = np.load("X_reduced_list.npz", allow_pickle=True)

imputation_1 = pd.read_parquet("imputation_1.parquet")

# Reconstruct as DataFrames (you'll need the original column names)
# Your full original feature list (80 features)
all_features = [
    'readmit_time_from_adm_m',
    'death_time_from_adm_m',
    'adm_age_rec3',
    'porc_pobr',
    'dit_m',
    'national_foreign',
    'ethnicity',
    'dg_psiq_cie_10_instudy',
    'dg_psiq_cie_10_dg',
    'dx_f3_mood',
    'dx_f6_personality',
    'dx_f_any_severe_mental',
    'any_phys_dx',
    'polysubstance_strict',
    'readmit_time_from_disch_m',      # ← outcome to exclude
    'readmit_event',                  # ← outcome to exclude
    'death_time_from_disch_m',        # ← outcome to exclude
    'death_event',                    # ← outcome to exclude
    'sex_rec_woman',
    'tenure_status_household_illegal_settlement',
    'tenure_status_household_owner_transferred_dwellings_pays_dividends',
    'tenure_status_household_renting',
    'tenure_status_household_stays_temporarily_with_a_relative',
    'cohabitation_alone',
    'cohabitation_with_couple_children',
    'cohabitation_family_of_origin',
    'sub_dep_icd10_status_drug_dependence',
    'any_violence_1_domestic_violence_sex_abuse',
    'prim_sub_freq_rec_2_2_6_days_wk',
    'prim_sub_freq_rec_3_daily',
    'tr_outcome_adm_discharge_adm_reasons',
    'tr_outcome_adm_discharge_rule_violation_undet',
    'tr_outcome_completion',
    'tr_outcome_dropout',
    'tr_outcome_referral',
    'adm_motive_another_sud_facility_fonodrogas_senda_previene',
    'adm_motive_justice_sector',
    'adm_motive_sanitary_sector',
    'adm_motive_spontaneous_consultation',
    'first_sub_used_alcohol',
    'first_sub_used_cocaine_paste',
    'first_sub_used_cocaine_powder',
    'first_sub_used_marijuana',
    'first_sub_used_opioids',
    'first_sub_used_tranquilizers_hypnotics',
    'primary_sub_mod_cocaine_paste',
    'primary_sub_mod_cocaine_powder',
    'primary_sub_mod_alcohol',
    'primary_sub_mod_marijuana',
    'tipo_de_vivienda_rec2_other_unknown',
    'plan_type_corr_m_pai',
    'plan_type_corr_m_pr',
    'plan_type_corr_pg_pai',
    'plan_type_corr_pg_pr',
    'occupation_condition_corr24_inactive',
    'occupation_condition_corr24_unemployed',
    'marital_status_rec_separated_divorced_annulled_widowed',
    'marital_status_rec_single',
    'urbanicity_cat_1_rural',
    'urbanicity_cat_2_mixed',
    'ed_attainment_corr_2_completed_high_school_or_less',
    'ed_attainment_corr_3_completed_primary_school_or_less',
    'evaluacindelprocesoteraputico_logro_intermedio',
    'evaluacindelprocesoteraputico_logro_minimo',
    'eva_consumo_logro_intermedio',
    'eva_consumo_logro_minimo',
    'eva_fam_logro_intermedio',
    'eva_fam_logro_minimo',
    'eva_relinterp_logro_intermedio',
    'eva_relinterp_logro_minimo',
    'eva_ocupacion_logro_intermedio',
    'eva_ocupacion_logro_minimo',
    'eva_sm_logro_intermedio',
    'eva_sm_logro_minimo',
    'eva_fisica_logro_intermedio',
    'eva_fisica_logro_minimo',
    'eva_transgnorma_logro_intermedio',
    'eva_transgnorma_logro_minimo',
    'id_centro'
]

outcome_indices = [
    all_features.index('readmit_time_from_disch_m'),
    all_features.index('readmit_time_from_adm_m'),
    all_features.index('readmit_event'),
    all_features.index('death_time_from_disch_m'),
    all_features.index('death_time_from_adm_m'),
    all_features.index('death_event'),
    all_features.index('id_centro')
]

# Combine all indices to remove
cols_to_remove = sorted(set(outcome_indices))

# Create new feature list WITHOUT these columns
feature_columns = [
    feat for i, feat in enumerate(all_features)
    if i not in cols_to_remove
]

# Now build X_reduced_list using ONLY the kept columns
X_reduced_list = [
    pd.DataFrame(data['X_reduced_list_1'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_2'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_3'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_4'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_5'][:, [i for i in range(len(all_features)) if i not in cols_to_remove]], columns=feature_columns)
]

# 3. “Best predictors” (variable importance) based on discrimination
* Inside each imputed dataset, we run k-fold CV, fit Coxnet on the training folds, and compute Uno’s C-index on the test folds.
* For each fold, we computed permutation importance by shuffling one predictor at a time in the test set, recomputing the C-index, and measuring the drop.
* We then pooled all these drops across folds and imputations, so `mean_drop_cindex` summarized how much that predictor hurts out-of-sample C-index on average, while respecting both multiple imputation and cross-validation.
* Sorting by `mean_drop_cindex` and taking the top 20 output the most influential predictors in a way that is robust to missing data and optimistic bias.


First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [37]:
import numpy as np
import pandas as pd

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Corrects Readmission time/event using Death data (Competing Risk).
    If Death happens BEFORE Readmission, we censor Readmission at the time of Death.
    """
    new_y_readm_list = []

    print("Correcting Readmission outcomes for Competing Risk (Death)...")

    for i, (y_r, y_d) in enumerate(zip(y_readm_list, y_death_list)):
        # Extract arrays
        r_time = y_r["time"]
        r_event = y_r["event"]

        d_time = y_d["time"]
        d_event = y_d["event"]

        # Create copies to modify
        new_r_time = r_time.copy()
        new_r_event = r_event.copy()

        # LOGIC:
        # Find cases where Death happened (d_event=1) AND Death was earlier than Readmission Time
        # Note: We compare times. If death is earlier, the patient never reached the "readmission" time potentially recorded.

        # Mask: Patients who died BEFORE the recorded readmission/censoring time
        # (or at the same time, if they died without being readmitted)
        mask_death_first = (d_event == True) & (d_time <= r_time)

        count_corrections = np.sum(mask_death_first)

        # 1. Update Time: Cut off at death time
        new_r_time[mask_death_first] = d_time[mask_death_first]

        # 2. Update Event: Force to 0 (Censored) because they died, not readmitted
        # (Unless they were readmitted exactly at the moment of death, which is unlikely/impossible in this context)
        new_r_event[mask_death_first] = False

        # Reconstruct structured array
        new_y_surv = np.empty(len(new_r_time), dtype=[("event", "?"), ("time", "<f8")])
        new_y_surv["event"] = new_r_event
        new_y_surv["time"] = new_r_time

        new_y_readm_list.append(new_y_surv)

        print(f"  Imputation {i+1}: Corrected {count_corrections} patients who died before readmission end-point.")

    return new_y_readm_list

# ==============================================================================
# EXECUTION
# ==============================================================================

# Create lists of outcomes (repeating the same observed outcome for each imputation)
# This is necessary because the function iterates over (X, y) pairs.
n_imputations = len(X_reduced_list)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

# Run this correction BEFORE fitting your models
y_surv_readm_list_corrected = correct_competing_risks(
    X_reduced_list,
    y_surv_readm_list,
    y_surv_death_list
)

# Now use `y_surv_readm_list_corrected` for your Readmission RSF/Cox models.

Correcting Readmission outcomes for Competing Risk (Death)...
  Imputation 1: Corrected 3203 patients who died before readmission end-point.
  Imputation 2: Corrected 3203 patients who died before readmission end-point.
  Imputation 3: Corrected 3203 patients who died before readmission end-point.
  Imputation 4: Corrected 3203 patients who died before readmission end-point.
  Imputation 5: Corrected 3203 patients who died before readmission end-point.


### Optimized version

In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed

def permutation_importance_cindex_cv_mi(
    X_list,
    y_surv_list, # Changed to receive a list of y_surv objects
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,  # New: number of parallel jobs (-1 = all cores)
):
    """
    Multiple-imputation + k-fold CV permutation importance for Coxnet
    using Uno's C-index (out-of-sample).

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed design matrices (one per imputation).
        All must have the same rows and columns in the same order.
    y_surv_list : list of structured arrays
        List of Surv(event, time) structured arrays (one per imputation).
    alpha_idx : int
        Index along the Coxnet regularization path to use for predictions.
    n_splits : int
        Number of CV folds.
    n_repeats : int
        Number of permutations per feature per fold.
    random_state : int
        Seed for KFold and permutations.
    Other kwargs : CoxnetSurvivalAnalysis hyperparameters.
    n_jobs : int, optional
        Number of jobs for parallel processing of imputation-fold combinations.

    Returns
    -------
    baseline_cindex_mean : float
        Mean baseline CV Uno C-index across folds and imputations.
    baseline_cindex_sd   : float
        SD of baseline CV Uno C-index across folds and imputations.
    df_imp : pandas.DataFrame
        Feature-level permutation importance pooled over folds & imputations:
        columns = [feature, mean_drop_cindex, sd_drop_cindex, n_evals]
    """
    # Convert to NumPy arrays upfront for speed
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(float) for X in X_list]
    n_imputations = len(X_list)
    n = X_list[0].shape[0]

    # Precompute CV splits once (same indices used for all imputations)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(n)))

    # Function to compute baseline C-index and drops for one imputation-fold
    def compute_fold(d, fold_idx, train_idx, test_idx):
        print(f"  Imputation {d}, fold {fold_idx+1}/{n_splits}")
        X_imp = X_list[d]
        X_train = X_imp[train_idx, :]
        X_test = X_imp[test_idx, :]
        y_train = y_surv_list[d][train_idx] # Access the correct y_surv for the current imputation
        y_test = y_surv_list[d][test_idx]   # Access the correct y_surv for the current imputation

        # Local RNG with unique seed for this fold-imputation
        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Effective alpha index
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)

        # Baseline risks and C-index
        risk_baseline = model.predict(X_test, alpha=model.alphas_[eff_alpha_idx])
        res_base = concordance_index_ipcw(y_train, y_test, risk_baseline)
        cindex_baseline = float(res_base[0])

        # Permutation drops per feature (list of lists)
        fold_drops = [[] for _ in range(n_features)]
        for col_idx in range(n_features):
            for r in range(n_repeats):
                X_perm = X_test.copy()
                X_perm[:, col_idx] = local_rng.permutation(X_perm[:, col_idx])

                risk_perm = model.predict(X_perm, alpha=model.alphas_[eff_alpha_idx])
                res_perm = concordance_index_ipcw(y_train, y_test, risk_perm)
                cindex_perm = float(res_perm[0])
                fold_drops[col_idx].append(cindex_baseline - cindex_perm)

        return cindex_baseline, fold_drops

    print(f"\n=== {n_imputations} imputations – {n_splits}-fold CV permutation importance ===")

    # Parallelize over all imputation-fold combinations
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Collect results
    baseline_cindices = [res[0] for res in results]
    global_drops = [[] for _ in range(n_features)]
    for res in results:
        fold_drops = res[1]
        for col_idx in range(n_features):
            global_drops[col_idx].extend(fold_drops[col_idx])

    # Aggregate feature importances
    imp_rows = []
    for col_idx in range(n_features):
        arr = np.array(global_drops[col_idx])
        mean_drop = float(arr.mean()) if arr.size > 0 else np.nan
        sd_drop = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
        imp_rows.append({
            "feature": feature_names[col_idx],
            "mean_drop_cindex": mean_drop,
            "sd_drop_cindex": sd_drop,
            "n_evals": int(arr.size),
        })

    df_imp = pd.DataFrame(imp_rows)
    df_imp = df_imp.sort_values("mean_drop_cindex", ascending=False).reset_index(drop=True)

    # Baseline C-index summary
    baseline_arr = np.array(baseline_cindices)
    baseline_cindex_mean = float(baseline_arr.mean())
    baseline_cindex_sd = float(baseline_arr.std(ddof=1)) if baseline_arr.size > 1 else 0.0

    print("\n=== Baseline CV Uno C-index over imputations & folds ===")
    print(f"Mean ± SD: {baseline_cindex_mean:.4f} ± {baseline_cindex_sd:.4f}")

    return baseline_cindex_mean, baseline_cindex_sd, df_imp

### Execute

This calls the function with your data and parameters, unpacking the three returned values into variables:
- **`baseline_cidx_readm`**: Mean baseline C-index across all CV folds and imputations (a float, e.g., 0.75). Indicates overall model performance without permutations, with higher means better risk ranking.
- `**baseline_cidx_sd_readm**`: Standard deviation of the baseline C-index (measures variability/reliability across runs).
- `**df_imp_readm**`: A Pandas DataFrame ranking features by importance (sorted descending by mean C-index decrease after permutation). Columns likely include: "feature": Feature name, "mean_decrease_cidx": Average C-index drop (higher = more important, as permutation hurts performance more), "sd_decrease_cidx": Standard deviation of the drop, "n_evals": Number of permutations run for that feature (e.g., n_imputations * n_splits * n_repeats).

**Arguments Explained** (similar to IBS version, but no times_eval since C-index doesn't require a time grid):
- **`X_list`**= X_reduced_list: List of imputed feature matrices (Pandas DataFrames). Each is a version of your predictors with missing values filled differently.
- **`y_surv`**= y_surv_readm: Survival target (structured NumPy array with 'event' bool and 'time' float fields). E.g., time to readmission or censoring.
- **`alpha_idx`**=25: Index in Coxnet's regularization path (alphas from strong to weak). Picks a model complexity level (lower index = sparser model).
- **`n_splits`**=5: CV folds (5-fold = 80% train/20% test per fold).
- **`n_repeats`**=5: Permutations per feature per fold per imputation. Higher = more stable importances but longer runtime.

**Defaults** (not shown but from definition): random_state=2125 (for reproducibility), Coxnet params like l1_ratio=0.9 (mostly Lasso for sparsity), and n_jobs=-1 (parallelize on all cores).

In [39]:
baseline_cidx_readm, baseline_cidx_sd_readm, df_imp_readm = (
    permutation_importance_cindex_cv_mi(
        X_list=X_reduced_list,
        y_surv_list=y_surv_readm_list_corrected, # Changed to use the corrected list
        alpha_idx=49,# el último alpha (menos regularización) #25,
        n_splits=5,
        n_repeats=5,        # maybe 3 for speed; you can increase
        # More flexible regularization:
        l1_ratio=0.1,    # ✅ Más Ridge (mantiene variables) .5         # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        alpha_min_ratio=0.001,# ✅ 100x más pequeño - CRÍTICO  #.1     # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)

# Top 20 predictors for readmission:
df_imp_readm.head(20)
#~3 hrs.


=== 5 imputations – 5-fold CV permutation importance ===


KeyboardInterrupt: 

In [ ]:
baseline_cidx_death, baseline_cidx_sd_death, df_imp_death = (
    permutation_importance_cindex_cv_mi(
        X_list=X_reduced_list,
        y_surv_list=y_surv_death_list, # Changed to use the list of death outcomes
        alpha_idx=25,
        n_splits=20,
        n_repeats=5,        # maybe 3 for speed; you can increase
                # More flexible regularization:
        l1_ratio= 0.1,#0.5,             # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        #0.1,         # Mostly Ridge (keeps features in, just shrinks them)
        alpha_min_ratio=.001,# Allow alpha to get very small (100x smaller than before)
        #0.1,      # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)

# Top 20 predictors for death:
df_imp_death.head(20)
#~3 hrs.

# Landmark

In [ ]:
# Tiempos de evaluación clínicamente relevantes
times_eval_grid = np.array([
    0.5,   # 2 semanas
    1,     # 1 mes
    3,     # 3 meses
    6,     # 6 meses
    12,    # 1 año
    36,    # 3 años
    60,    # 5 años
    120    # 10 años (si tienes seguimiento tan largo)
])

# Filtrar solo tiempos dentro de tu rango de datos
max_time = np.max([y['time'].max() for y in y_surv_death_list])
times_eval_grid = times_eval_grid[times_eval_grid <= max_time]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, brier_score
from joblib import Parallel, delayed

def permutation_importance_timespecific_cv_mi(
    X_list,
    y_surv_list,
    times_eval,  # New: array of time points to evaluate
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.1,
    alpha_min_ratio=0.001,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Multiple-imputation + k-fold CV permutation importance for Coxnet
    with time-specific performance metrics.

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed design matrices (one per imputation).
    y_surv_list : list of structured arrays
        List of Surv(event, time) structured arrays (one per imputation).
    times_eval : array-like
        Time points at which to evaluate performance (in same units as time in y_surv).
    alpha_idx : int
        Index along the Coxnet regularization path to use for predictions.
    n_splits : int
        Number of CV folds.
    n_repeats : int
        Number of permutations per feature per fold.
    random_state : int
        Seed for KFold and permutations.
    l1_ratio : float
        ElasticNet mixing parameter.
    alpha_min_ratio : float
        Minimum alpha as fraction of maximum alpha.
    n_alphas : int
        Number of alphas along regularization path.
    max_iter : int
        Maximum iterations for model fitting.
    n_jobs : int
        Number of parallel jobs (-1 = all cores).

    Returns
    -------
    df_global_metrics : pandas.DataFrame
        Global performance metrics (C-index, mean AUC, mean BS).
    df_time_metrics : pandas.DataFrame
        Time-specific metrics (AUC(t), BS(t)) for each time point.
    df_importance : pandas.DataFrame
        Feature-level permutation importance (C-index based).
    """
    # Convert to NumPy arrays upfront for speed
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(float) for X in X_list]
    n_imputations = len(X_list)
    n = X_list[0].shape[0]

    # Filter times_eval to be within data range
    max_time = np.max([y['time'].max() for y in y_surv_list])
    times_eval = np.array(times_eval)
    times_eval = times_eval[times_eval <= max_time]
    n_times = len(times_eval)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(n)))

    # Storage for time-specific metrics
    results_by_time = {t: {'auc': [], 'bs': []} for t in times_eval}

    def format_time_label(months):
        """Convert months to readable label."""
        if months < 1:
            return f"{int(months*4)}wk"
        elif months < 12:
            return f"{int(months)}mo"
        else:
            return f"{int(months/12)}yr"

    # Function to compute all metrics for one imputation-fold
    def compute_fold(d, fold_idx, train_idx, test_idx):
        print(f"  Imputation {d+1}/{n_imputations}, fold {fold_idx+1}/{n_splits}")
        X_imp = X_list[d]
        X_train = X_imp[train_idx, :]
        X_test = X_imp[test_idx, :]
        y_train = y_surv_list[d][train_idx]
        y_test = y_surv_list[d][test_idx]

        # Local RNG with unique seed
        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Effective alpha index
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)

        # Baseline predictions
        risk_baseline = model.predict(X_test, alpha=model.alphas_[eff_alpha_idx])

        # 1. Global C-index
        res_base = concordance_index_ipcw(y_train, y_test, risk_baseline)
        cindex_baseline = float(res_base[0])

        # 2. Time-specific AUC
        try:
            auc_scores, mean_auc = cumulative_dynamic_auc(
                y_train, y_test, risk_baseline, times=times_eval
            )
        except Exception as e:
            print(f"    Warning: AUC calculation failed - {str(e)}")
            auc_scores = np.full(n_times, np.nan)

        # 3. Time-specific Brier Score
        bs_scores = []
        for t in times_eval:
            try:
                _, bs_t = brier_score(y_train, y_test, risk_baseline, times=[t])
                bs_scores.append(bs_t[0])
            except Exception as e:
                print(f"    Warning: BS calculation failed at t={t} - {str(e)}")
                bs_scores.append(np.nan)
        bs_scores = np.array(bs_scores)

        # 4. Permutation importance (C-index based)
        fold_drops = [[] for _ in range(n_features)]
        for col_idx in range(n_features):
            for r in range(n_repeats):
                X_perm = X_test.copy()
                X_perm[:, col_idx] = local_rng.permutation(X_perm[:, col_idx])

                risk_perm = model.predict(X_perm, alpha=model.alphas_[eff_alpha_idx])
                res_perm = concordance_index_ipcw(y_train, y_test, risk_perm)
                cindex_perm = float(res_perm[0])
                fold_drops[col_idx].append(cindex_baseline - cindex_perm)

        return {
            'cindex': cindex_baseline,
            'auc_scores': auc_scores,
            'bs_scores': bs_scores,
            'fold_drops': fold_drops
        }

    print(f"\n{'='*80}")
    print(f"TIME-SPECIFIC PERFORMANCE EVALUATION")
    print(f"{'='*80}")
    print(f"Imputations: {n_imputations}, CV folds: {n_splits}")
    print(f"Time points: {len(times_eval)}")
    print(f"Times (months): {times_eval}")
    print(f"{'='*80}\n")

    # Parallelize over all imputation-fold combinations
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Collect global C-index
    baseline_cindices = [res['cindex'] for res in results]

    # Collect time-specific metrics
    all_auc_scores = np.array([res['auc_scores'] for res in results])  # shape: (n_folds, n_times)
    all_bs_scores = np.array([res['bs_scores'] for res in results])

    # Collect permutation importance
    global_drops = [[] for _ in range(n_features)]
    for res in results:
        fold_drops = res['fold_drops']
        for col_idx in range(n_features):
            global_drops[col_idx].extend(fold_drops[col_idx])

    # ==================== AGGREGATE GLOBAL METRICS ====================
    cindex_mean = np.mean(baseline_cindices)
    cindex_sd = np.std(baseline_cindices, ddof=1) if len(baseline_cindices) > 1 else 0.0

    # Mean AUC across all times and folds
    mean_auc_global = np.nanmean(all_auc_scores)
    sd_auc_global = np.nanstd(all_auc_scores, ddof=1)

    # Mean BS across all times and folds
    mean_bs_global = np.nanmean(all_bs_scores)
    sd_bs_global = np.nanstd(all_bs_scores, ddof=1)

    df_global = pd.DataFrame([{
        'cindex_mean': cindex_mean,
        'cindex_sd': cindex_sd,
        'auc_mean_global': mean_auc_global,
        'auc_sd_global': sd_auc_global,
        'bs_mean_global': mean_bs_global,
        'bs_sd_global': sd_bs_global,
        'n_folds_total': len(baseline_cindices),
        'n_timepoints': n_times
    }])

    # ==================== AGGREGATE TIME-SPECIFIC METRICS ====================
    time_rows = []
    for i, t in enumerate(times_eval):
        auc_at_t = all_auc_scores[:, i]
        bs_at_t = all_bs_scores[:, i]

        time_rows.append({
            'time_months': t,
            'time_label': format_time_label(t),
            'auc_mean': np.nanmean(auc_at_t),
            'auc_sd': np.nanstd(auc_at_t, ddof=1),
            'auc_min': np.nanmin(auc_at_t),
            'auc_max': np.nanmax(auc_at_t),
            'bs_mean': np.nanmean(bs_at_t),
            'bs_sd': np.nanstd(bs_at_t, ddof=1),
            'bs_min': np.nanmin(bs_at_t),
            'bs_max': np.nanmax(bs_at_t),
            'n_evals': np.sum(~np.isnan(auc_at_t))
        })

    df_time = pd.DataFrame(time_rows)

    # ==================== AGGREGATE FEATURE IMPORTANCE ====================
    imp_rows = []
    for col_idx in range(n_features):
        arr = np.array(global_drops[col_idx])
        mean_drop = float(arr.mean()) if arr.size > 0 else np.nan
        sd_drop = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
        imp_rows.append({
            "feature": feature_names[col_idx],
            "mean_drop_cindex": mean_drop,
            "sd_drop_cindex": sd_drop,
            "n_evals": int(arr.size),
        })

    df_imp = pd.DataFrame(imp_rows)
    df_imp = df_imp.sort_values("mean_drop_cindex", ascending=False).reset_index(drop=True)

    # ==================== PRINT SUMMARY ====================
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"\n>>> GLOBAL METRICS (averaged across all times) <<<")
    print(f"C-index:       {cindex_mean:.4f} ± {cindex_sd:.4f}")
    print(f"Mean AUC(t):   {mean_auc_global:.4f} ± {sd_auc_global:.4f}")
    print(f"Mean BS(t):    {mean_bs_global:.4f} ± {sd_bs_global:.4f}")

    print(f"\n>>> TIME-SPECIFIC METRICS <<<")
    print(df_time[['time_label', 'auc_mean', 'auc_sd', 'bs_mean', 'bs_sd']].to_string(index=False))

    print(f"\n>>> TOP 10 MOST IMPORTANT FEATURES (by C-index drop) <<<")
    print(df_imp.head(10).to_string(index=False))
    print(f"{'='*80}\n")

    return df_global, df_time, df_imp


# ==================== USAGE EXAMPLE ====================
if __name__ == "__main__":
    # Define time grid
    times_eval_grid = np.array([
        0.5,   # 2 weeks
        1,     # 1 month
        3,     # 3 months
        6,     # 6 months
        12,    # 1 year
        36,    # 3 years
        60,    # 5 years
        120    # 10 years
    ])

    # Run analysis
    df_global, df_time, df_importance = permutation_importance_timespecific_cv_mi(
        X_list=X_reduced_list,
        y_surv_list=y_surv_readm_list_corrected,
        times_eval=times_eval_grid,
        alpha_idx=49,  # Use last alpha (least regularization)
        n_splits=5,
        n_repeats=20,
        random_state=2125,
        # Corrected parameters to avoid model collapse
        l1_ratio=0.1,              # More Ridge
        alpha_min_ratio=0.001,     # Allow smaller alphas
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )

    # Display results
    print("\n" + "="*80)
    print("DETAILED GLOBAL METRICS")
    print("="*80)
    print(df_global.T)

    print("\n" + "="*80)
    print("DETAILED TIME-SPECIFIC PERFORMANCE")
    print("="*80)
    print(df_time)

    print("\n" + "="*80)
    print("TOP 20 FEATURES")
    print("="*80)
    print(df_importance.head(20))

In [ ]:
# Run analysis
df_global_death, df_time_death, df_importance_death = permutation_importance_timespecific_cv_mi(
  X_list=X_reduced_list,
  y_surv_list=y_surv_death_list,
  times_eval=times_eval_grid,
  alpha_idx=49,  # Use last alpha (least regularization)
  n_splits=10,
  n_repeats=20,
  random_state=2125,
  # Corrected parameters to avoid model collapse
  l1_ratio=0.1,              # More Ridge
  alpha_min_ratio=0.001,     # Allow smaller alphas
  n_alphas=50,
  max_iter=100000,
  n_jobs=-1
)

# Display results
print("\n" + "="*80)
print("DETAILED GLOBAL METRICS")
print("="*80)
print(df_global_death.T)

print("\n" + "="*80)
print("DETAILED TIME-SPECIFIC PERFORMANCE")
print("="*80)
print(df_time_death)

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)
print(df_importance_death.head(20))

### Visualize

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_time_dependent_performance(df_time, outcome_name="Readmission"):
    """
    Visualize time-specific AUC and Brier Score.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: AUC(t) with confidence bands
    axes[0].errorbar(
        df_time['time_months'],
        df_time['auc_mean'],
        yerr=df_time['auc_sd'],
        marker='o',
        markersize=8,
        capsize=6,
        linewidth=2.5,
        color='steelblue',
        ecolor='lightblue',
        label='Mean AUC(t) ± SD'
    )
    axes[0].fill_between(
        df_time['time_months'],
        df_time['auc_min'],
        df_time['auc_max'],
        alpha=0.15,
        color='steelblue',
        label='Min-Max Range'
    )
    axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random Chance')
    axes[0].axhline(0.7, color='green', linestyle=':', linewidth=1.5, label='Good Discrimination')
    axes[0].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('AUC(t)', fontsize=13, fontweight='bold')
    axes[0].set_title(f'Time-Dependent Discrimination - {outcome_name}', fontsize=15, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3, linestyle='--')
    axes[0].set_ylim([0.4, 1.0])

    # Add time labels on x-axis
    axes[0].set_xticks(df_time['time_months'])
    axes[0].set_xticklabels(df_time['time_label'], rotation=45, ha='right')

    # Plot 2: Brier Score(t)
    axes[1].errorbar(
        df_time['time_months'],
        df_time['bs_mean'],
        yerr=df_time['bs_sd'],
        marker='s',
        markersize=8,
        capsize=6,
        linewidth=2.5,
        color='darkorange',
        ecolor='bisque',
        label='Mean BS(t) ± SD'
    )
    axes[1].fill_between(
        df_time['time_months'],
        df_time['bs_min'],
        df_time['bs_max'],
        alpha=0.15,
        color='darkorange',
        label='Min-Max Range'
    )
    axes[1].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Brier Score(t)', fontsize=13, fontweight='bold')
    axes[1].set_title(f'Time-Dependent Calibration - {outcome_name}', fontsize=15, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3, linestyle='--')

    # Add time labels
    axes[1].set_xticks(df_time['time_months'])
    axes[1].set_xticklabels(df_time['time_label'], rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

# Use it
plot_time_dependent_performance(df_time, outcome_name="Readmission")

In [ ]:
def plot_time_dependent_performance(df_time, outcome_name="Mortality"):
    """
    Visualize time-specific AUC and Brier Score.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: AUC(t) with confidence bands
    axes[0].errorbar(
        df_time_death['time_months'],
        df_time_death['auc_mean'],
        yerr=df_time_death['auc_sd'],
        marker='o',
        markersize=8,
        capsize=6,
        linewidth=2.5,
        color='steelblue',
        ecolor='lightblue',
        label='Mean AUC(t) ± SD'
    )
    axes[0].fill_between(
        df_time_death['time_months'],
        df_time_death['auc_min'],
        df_time_death['auc_max'],
        alpha=0.15,
        color='steelblue',
        label='Min-Max Range'
    )
    axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random Chance')
    axes[0].axhline(0.7, color='green', linestyle=':', linewidth=1.5, label='Good Discrimination')
    axes[0].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('AUC(t)', fontsize=13, fontweight='bold')
    axes[0].set_title(f'Time-Dependent Discrimination - {outcome_name}', fontsize=15, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3, linestyle='--')
    axes[0].set_ylim([0.4, 1.0])

    # Add time labels on x-axis
    axes[0].set_xticks(df_time_death['time_months'])
    axes[0].set_xticklabels(df_time_death['time_label'], rotation=45, ha='right')

    # Plot 2: Brier Score(t)
    axes[1].errorbar(
        df_time_death['time_months'],
        df_time_death['bs_mean'],
        yerr=df_time_death['bs_sd'],
        marker='s',
        markersize=8,
        capsize=6,
        linewidth=2.5,
        color='darkorange',
        ecolor='bisque',
        label='Mean BS(t) ± SD'
    )
    axes[1].fill_between(
        df_time_death['time_months'],
        df_time_death['bs_min'],
        df_time_death['bs_max'],
        alpha=0.15,
        color='darkorange',
        label='Min-Max Range'
    )
    axes[1].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Brier Score(t)', fontsize=13, fontweight='bold')
    axes[1].set_title(f'Time-Dependent Calibration - {outcome_name}', fontsize=15, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3, linestyle='--')

    # Add time labels
    axes[1].set_xticks(df_time_death['time_months'])
    axes[1].set_xticklabels(df_time_death['time_label'], rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

# Use it
plot_time_dependent_performance(df_time, outcome_name="Mortality")